# YOLO26 Fine-Tuning — Kaggle / Colab / Local

Fine-tunes **YOLO26** on the Roboflow `animals-kapzz` v4 dataset.
Works on Kaggle (2× T4, **önerilen**), Google Colab (T4) and local machines (Apple Silicon / CUDA / CPU).

---

**Kaggle ile başlangıç (2× T4 — ücretsiz, ~20 dk):**
1. `+ Create` → `New Notebook` → `File → Import Notebook` → `yolo_train.ipynb`
2. Sağ panel → **Session options** → Accelerator: **GPU T4 x2**
3. Sağ panel → **Session options** → Internet: **On** (Roboflow indirmesi için gerekli)
4. Tüm hücreleri çalıştır.
5. Eğitim bitince sağ panel → **Output** sekmesi → `yolo26_animals.zip` → indir.

---

**Colab (tek T4, ~40 dk):**
1. `File → Upload notebook` → `yolo_train.ipynb`
2. `Runtime → Change runtime type → T4 GPU`
3. Tüm hücreleri çalıştır. Son hücre `yolo26_animals.zip`'i tarayıcıya indirir.

---

**Local (Apple M-series / CUDA / CPU):** tüm hücreleri çalıştır. M4'te YOLO26-s için ~6 sa.

## 1. Detect environment

Auto-detects Colab vs local and picks the right device.

In [ ]:
import sys, os, pathlib

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
print('Running in Colab :', IN_COLAB)
print('Running in Kaggle:', IN_KAGGLE)

if IN_COLAB:
    WORK_DIR = pathlib.Path('/content')
elif IN_KAGGLE:
    WORK_DIR = pathlib.Path('/kaggle/working')
else:
    WORK_DIR = pathlib.Path.cwd()

os.chdir(WORK_DIR)
print('Working directory :', WORK_DIR)

## 2. Install dependencies

Colab comes with torch/ultralytics but may be older — we pin the required version.

In [ ]:
%pip install --quiet 'ultralytics>=8.4.41' 'roboflow>=1.1.40' pyyaml

## 3. Download dataset (cached)

Paste your Roboflow API key below (or set the `ROBOFLOW_API_KEY` env var). Dataset
downloads only if `./datasets/Animals-4/` does not already exist.

In [ ]:
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'lpnLA6gQ9PQq5C0pGjuV')
DATASET_DIR = (WORK_DIR / 'datasets' / 'Animals-4').resolve()

if not DATASET_DIR.exists():
    from roboflow import Roboflow
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    os.chdir(DATASET_DIR.parent)
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    rf.workspace('graduation-nnzal').project('animals-kapzz').version(4).download('yolov8')
    os.chdir(WORK_DIR)
    print('Dataset downloaded to', DATASET_DIR)
else:
    print('Dataset already exists at', DATASET_DIR)

print('Contents:', [p.name for p in DATASET_DIR.iterdir()])

## 4. Normalize `data.yaml` paths

Roboflow writes hard-coded paths; we rewrite them to point to the local folder.

In [ ]:
import yaml

data_yaml = DATASET_DIR / 'data.yaml'
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

cfg['path']  = str(DATASET_DIR)
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
if (DATASET_DIR / 'test/images').exists():
    cfg['test'] = 'test/images'

with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Rewrote', data_yaml)
print('Classes:', cfg.get('names'))
print('nc     :', cfg.get('nc'))

## 5. Train YOLO26

**Device is auto-selected:** CUDA on Colab, MPS on Apple Silicon, CPU otherwise.

Model size:
- `yolo26n.pt` → fastest, lowest accuracy
- `yolo26s.pt` → **recommended** (good accuracy/speed trade-off)
- `yolo26m.pt` → highest accuracy; reduce `batch` to 8 on Colab T4 if OOM

On Colab T4: batch 16, YOLO26-s, 100 epochs ≈ 35-45 min.

In [ ]:
from ultralytics import YOLO
import torch

MODEL  = 'yolo26s.pt'
EPOCHS = 100
BATCH  = 16  # GPU başına; 2x T4'te efektif batch = 32

gpu_count = torch.cuda.device_count()

if gpu_count >= 2:
    DEVICE = list(range(gpu_count))
    print(f'Using {gpu_count}x GPU:', [torch.cuda.get_device_name(i) for i in DEVICE])
elif gpu_count == 1:
    DEVICE = 0
    print('Using CUDA GPU:', torch.cuda.get_device_name(0))
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('Using Apple MPS')
else:
    DEVICE = 'cpu'
    print('Using CPU — training will be very slow')

WORKERS = 4 if (IN_COLAB or IN_KAGGLE) else 2

model = YOLO(MODEL)

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    project='runs',
    name='yolo26_animals',
    exist_ok=True,
    patience=20,
    cache='disk',
    workers=WORKERS,
    optimizer='AdamW',
    lr0=1e-3,
    seed=0,
)

SAVE_DIR = pathlib.Path(results.save_dir)
print('Weights saved to:', SAVE_DIR)

## 6. Validate

Reports mAP50, mAP50-95, precision, recall on the validation split.

In [ ]:
metrics = model.val(data=str(data_yaml), imgsz=640, device=DEVICE)
print('mAP50     :', float(metrics.box.map50))
print('mAP50-95  :', float(metrics.box.map))

## 7. Export

- ONNX: cross-platform, works on your local machine via onnxruntime.
- CoreML: native Apple Neural Engine acceleration (skipped if unavailable).
- TorchScript: fallback.

In [ ]:
best_pt = SAVE_DIR / 'weights' / 'best.pt'
print('best.pt:', best_pt, 'exists:', best_pt.exists())

best_model = YOLO(str(best_pt))
best_model.export(format='onnx')

try:
    best_model.export(format='coreml')
except Exception as e:
    print('CoreML export skipped:', e)

## 8. Download trained weights

**Kaggle:** `yolo26_animals.zip` dosyası `/kaggle/working/` klasörüne kaydedilir.
Sağ paneldeki **Output** sekmesinden indirilebilir (klasör ikonu).

**Colab:** Son hücre `yolo26_animals.zip`'i otomatik olarak tarayıcıya indirir.

**Local:** Dosyalar zaten diskinizde — `SAVE_DIR` çıktısına bakın.

In [ ]:
import shutil

out_dir = SAVE_DIR
archive = WORK_DIR / 'yolo26_animals.zip'

if archive.exists():
    archive.unlink()
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=out_dir.parent, base_dir=out_dir.name)
print('Created', archive, archive.stat().st_size // 1024, 'KB')

if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
elif IN_KAGGLE:
    print('Kaggle: sağ paneldeki Output sekmesinden yolo26_animals.zip dosyasını indir.')
else:
    print('Local run — archive saved at', archive)

## 9. Use the new weights in the repo

After downloading `yolo26_animals.zip` to your laptop, unzip it into the project root:

```bash
unzip -o yolo26_animals.zip -d yolomodel/
# → yolomodel/yolo26_animals/weights/best.pt
```

Then either:

**(a) One-off test:**
```bash
UAV_MODEL=yolomodel/yolo26_animals/weights/best.pt python videoproc_v3_video.py
```

**(b) Permanent default** — edit `config.py`:
```python
FINE_TUNED_MODEL = 'yolomodel/yolo26_animals/weights/best.pt'
```